# Exemplo: Regressão
---------------------

Este exemplo mostra como usar o ExperionML para aplicar PCA aos dados e executar um pipeline de regressão.

Baixe o conjunto de dados abalone em [https://archive.ics.uci.edu/ml/datasets/Abalone](https://archive.ics.uci.edu/ml/datasets/Abalone). O objetivo deste conjunto de dados é prever o número de anéis, isto é, a idade, das conchas de abalone a partir de medições físicas.

### Carregar os dados

In [1]:
# Import packages
import pandas as pd
from experionml import ExperionMLRegressor

In [2]:
# Carregar os dados
X = pd.read_csv("./datasets/abalone.csv")

# Let's have a look
X.head()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [3]:
# Inicialize o experionml para tarefas de regressão
experionml = ExperionMLRegressor(X, "Rings", verbose=2, random_state=42)

<< ================== ExperionML ================== >>

Configuração ==================== >>
Tarefa do algoritmo: Regression.

Estatísticas do conjunto de dados ==================== >>
Formato: (4177, 9)
Tamanho do conjunto de train: 3342
Tamanho do conjunto de test: 835
-------------------------------------
Memória: 300.87 kB
Escalonado: False
Atributos categóricos: 1 (12.5%)
Valores atípicos: 195 (0.6%)



In [4]:
# Codifique as variáveis categóricas
experionml.encode()

Ajustando Encoder...
Codificando colunas categóricas...
 --> Aplicando OneHot-encoding à variável Sex. Ela contém 3 classes.


In [5]:
# Plote a matriz de correlação do conjunto de dados
experionml.plot_correlation()

In [6]:
# Apply pca for dimensionality reduction
experionml.feature_selection(strategy="pca", n_features=6)

Fitting FeatureSelector...
Performing feature selection ...
 --> Applying Principal Component Analysis...
   --> Scaling features...
   --> Keeping 6 components.
   --> Explained variance ratio: 0.97


In [7]:
# Observe que as variáveis são renomeadas automaticamente para pca0, pca1 etc.
experionml.columns

Index(['pca0', 'pca1', 'pca2', 'pca3', 'pca4', 'pca5', 'Rings'], dtype='object')

In [8]:
# Use os métodos de plotagem para ver a razão de variância retida
experionml.plot_pca()

In [9]:
experionml.plot_components()

### Executar o pipeline

In [10]:
experionml.run(
    models=["Tree", "Bag", "ET"],
    metric="mse",
    n_trials=5,
    n_bootstrap=5,
)


Training ========================= >>
Models: Tree, Bag, ET
Metric: mse


Running hyperparameter tuning for DecisionTree...
| trial |   criterion | splitter | max_depth | min_samples_split | min_samples_leaf | max_features | ccp_alpha |     mse | best_mse | time_trial | time_ht |    state |
| ----- | ----------- | -------- | --------- | ----------------- | ---------------- | ------------ | --------- | ------- | -------- | ---------- | ------- | -------- |


| 0     | absolute_.. |     best |         5 |                 8 |               10 |         None |     0.035 | -6.5456 |  -6.5456 |     0.247s |  0.247s | COMPLETE |
| 1     | squared_e.. |     best |        10 |                 5 |                1 |          0.5 |      0.03 | -5.9179 |  -5.9179 |     0.071s |  0.318s | COMPLETE |
| 2     | absolute_.. |   random |        14 |                15 |               16 |         sqrt |     0.025 | -9.1674 |  -5.9179 |     0.080s |  0.398s | COMPLETE |


| 3     | friedman_.. |   random |         4 |                10 |               17 |          0.9 |      0.01 |  -7.856 |  -5.9179 |     0.068s |  0.466s | COMPLETE |
| 4     |     poisson |     best |        12 |                15 |                8 |          0.6 |      0.02 | -6.7356 |  -5.9179 |     0.089s |  0.556s | COMPLETE |
Hyperparameter tuning ---------------------------
Best trial --> 1
Best parameters:
 --> criterion: squared_error
 --> splitter: best
 --> max_depth: 10
 --> min_samples_split: 5
 --> min_samples_leaf: 1
 --> max_features: 0.5
 --> ccp_alpha: 0.03
Best evaluation --> mse: -5.9179
Time elapsed: 0.556s
Fit ---------------------------------------------
Train evaluation --> mse: -5.09
Test evaluation --> mse: -6.6133
Time elapsed: 0.050s


Bootstrap ---------------------------------------
Evaluation --> mse: -6.9915 ± 0.3004
Time elapsed: 0.096s
-------------------------------------------------
Time: 0.702s


Running hyperparameter tuning for Bagging...
| trial | n_estimators | max_samples | max_features | bootstrap | bootstrap_features |     mse | best_mse | time_trial | time_ht |    state |
| ----- | ------------ | ----------- | ------------ | --------- | ------------------ | ------- | -------- | ---------- | ------- | -------- |


| 0     |          190 |         1.0 |          0.9 |      True |               True |  -4.597 |   -4.597 |     2.803s |  2.803s | COMPLETE |


...

| 4     |          340 | squared_error |         6 |                15 |                8 |         None |      True |         0.8 |      0.01 | -5.1464 |  -4.7733 |     0.528s |  4.828s | COMPLETE |
Hyperparameter tuning ---------------------------
Best trial --> 3
Best parameters:
 --> n_estimators: 100
 --> criterion: squared_error
 --> max_depth: 14
 --> min_samples_split: 15
 --> min_samples_leaf: 8
 --> max_features: None
 --> bootstrap: True
 --> max_samples: 0.9
 --> ccp_alpha: 0.005
Best evaluation --> mse: -4.7733
Time elapsed: 4.828s
Fit ---------------------------------------------


Train evaluation --> mse: -4.6866
Test evaluation --> mse: -5.9301
Time elapsed: 0.227s


Bootstrap ---------------------------------------
Evaluation --> mse: -6.0381 ± 0.082
Time elapsed: 0.931s
-------------------------------------------------
Time: 5.986s


Resultados finais ==================== >>
Tempo total: 36.744s
-------------------------------------


DecisionTree --> mse: -6.9915 ± 0.3004 ~


Bagging      --> mse: -5.8869 ± 0.1089 ~ !
ExtraTrees   --> mse: -6.0381 ± 0.082 ~


### Analisar os resultados

In [11]:
# Use os gráficos de erros ou resíduos para verificar o desempenho dos modelos
experionml.plot_residuals()

In [12]:
experionml.plot_errors()

In [13]:
# Analise a relação entre a resposta alvo e as variáveis
experionml.plot_partial_dependence(columns=(0, 1, 2, 3))